# Sesión 6 · Práctica guiada: explorar un barómetro paso a paso

Esta práctica la hacemos juntos, paso a paso: el profesor escribe y explica cada celda, y tú la reproduces en tu notebook.

Vais a preparar un barómetro con una cadena de funciones pequeñas y a responder con ella una pregunta: **¿a quién le preocupa más la vivienda?**

> Por defecto se usa `datos/barometro.sav`, que es **simulado**. Si habéis descargado un barómetro real de la web del CIS, cambiad la ruta y adaptad los nombres de las variables.

In [ ]:
# Preparacion: localiza la carpeta de la asignatura (y la descarga si estas en Google Colab).
# Ejecuta esta celda primero; no hace falta entenderla todavia.
import os, subprocess
from pathlib import Path

REPO = "https://github.com/luisaugustos/ppy-analisis-social.git"


def _buscar_carpeta():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        for candidata in (carpeta, carpeta / "ppy-analisis-social"):
            if (candidata / "datos" / "ORIGEN.md").is_file():
                return candidata
    return None


raiz = _buscar_carpeta()
if raiz is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    raiz = Path("ppy-analisis-social").resolve()

os.chdir(raiz)
print("Carpeta de trabajo lista. Los datos estan en la subcarpeta 'datos'.")

In [ ]:
import numpy as np
import pandas as pd
import pyreadstat

barometro, meta = pyreadstat.read_sav("datos/barometro.sav")
print(barometro.shape)
meta.column_names_to_labels

## Ejercicio 1 · Leer las etiquetas

Mostrad las etiquetas de los valores de `ESTUDIOS` y de `SITECON`. ¿Qué códigos de `SITECON` corresponden a «no sabe» y «no contesta»?

In [ ]:
# TODO 1
...

## Ejercicio 2 · Paso «quitar la no respuesta»

Escribid `quitar_no_respuesta(tabla, variable, codigos)`: devuelve una **copia** de la tabla en la que los códigos indicados de esa variable pasan a ser valores perdidos.

Pista: `tabla[variable].where(condicion)` conserva los valores que cumplen la condición y deja perdidos los demás.

In [ ]:
# TODO 2
...


prueba = quitar_no_respuesta(barometro, "SITECON", [8, 9])
assert prueba["SITECON"].max() == 5 and barometro["SITECON"].max() == 9      # la original no cambia

## Ejercicio 3 · Paso «poner etiquetas»

Escribid `poner_etiquetas(tabla, meta, variables)`: devuelve una copia en la que los códigos de esas variables se sustituyen por sus etiquetas.

In [ ]:
# TODO 3
...


assert set(poner_etiquetas(barometro, meta, ["SEXO"])["SEXO"]) == {"Hombre", "Mujer"}

## Ejercicio 4 · Paso «grupo de edad»

Escribid `anadir_grupo_edad(tabla)`: añade la columna `grupo_edad` con los tramos 18-34, 35-49, 50-64 y 65 y más (`pd.cut`).

In [ ]:
# TODO 4
...


assert anadir_grupo_edad(barometro)["grupo_edad"].notna().all()

## Ejercicio 5 · Encadenar

Construid `encuesta` encadenando vuestros pasos con `.pipe`: quitad la no respuesta de `SITECON` (códigos 8 y 9) y de `ESCIDEOL` (98 y 99), añadid el grupo de edad y poned etiquetas a `SEXO`, `ESTUDIOS` y `TENENCIA`.

In [ ]:
# TODO 5
...

assert "grupo_edad" in encuesta.columns and encuesta["ESCIDEOL"].max() <= 10
encuesta.head(3)

## Ejercicio 6 · ¿Importan los pesos?

Calculad el porcentaje que menciona la vivienda como problema (`PROBVIV`) sin ponderar y ponderado.

In [ ]:
# TODO 6
...

print(f"Sin ponderar: {sin_pesos:.1f} %  ·  Ponderado: {con_pesos:.1f} %")
assert sin_pesos != con_pesos

## Ejercicio 7 · Una función para cualquier grupo

Completad `porcentaje_ponderado`, que devuelve una tabla con el porcentaje ponderado de una variable 0/1 y el número de entrevistas de cada grupo.

In [ ]:
def porcentaje_ponderado(tabla, variable, grupo, peso="PESO"):
    """Porcentaje ponderado de una variable 0/1 por grupo, con el número de entrevistas."""
    filas = []
    for nombre, datos in tabla.groupby(grupo, observed=True):
        # TODO 7: calculad el porcentaje ponderado de 'variable' dentro de 'datos'
        ...
        filas.append({grupo: nombre, "porcentaje": round(porcentaje, 1), "entrevistas": len(datos)})
    return pd.DataFrame(filas)


por_edad = porcentaje_ponderado(encuesta, "PROBVIV", "grupo_edad")
assert len(por_edad) == 4 and por_edad["entrevistas"].sum() == len(encuesta)
por_edad

## Ejercicio 8 · Responder a la pregunta

Usad vuestra función para ver el porcentaje que menciona la vivienda según el régimen de tenencia y según el nivel de estudios. Dibujad uno de los dos resultados con un gráfico de barras horizontales.

In [ ]:
# TODO 8
...

## Ejercicio 9 · ¿Quién no contesta?

Cread la columna `sin_respuesta`, que valga 1 si falta `SITECON` y 0 si no, y usad `porcentaje_ponderado` para ver si la no respuesta cambia con la edad.

In [ ]:
# TODO 9
...

no_respuesta

## Ejercicio 10 · Guardar vuestras funciones

Guardad los tres pasos y `porcentaje_ponderado` en un módulo `mi_encuesta.py` con `%%writefile`, como hicimos en la sesión 4, e importadlo en una celda nueva. (Sin solución: es copiar vuestras funciones.)

## Para cerrar

- ¿Qué afirmaciones de hoy sostendríais en un informe y cuáles son solo pistas?
- **Entregable 2 (sesiones 5–6):** se publica hoy. Consultad `entregables/entregable_2.md`.
- **Tras las vacaciones (13 de enero): tutoría grupal.** Traed la propuesta de proyecto: pregunta, fuentes y plan (`proyecto/guia_proyecto.md`).